# MNIST Flow Matching 生成示例
本 Notebook 使用 Flow Matching 在 MNIST 上训练速度场模型，并通过数值积分从高斯噪声生成手写数字。


## 1) 环境检查与依赖准备
这段代码会检查 `torch`、`torchvision`、`matplotlib`、`tqdm` 是否可用；若缺失则尝试自动安装，避免后续单元报错。


In [ ]:
import importlib.util
import subprocess
import sys


def ensure_packages(packages):
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if not missing:
        print("???????")
        return True

    print(f"???????: {missing}???????...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        print("???????")
        return True
    except Exception as e:
        print("????????????????")
        print(f"????: {e}")
        return False

DEPENDENCIES_READY = ensure_packages(["torch", "torchvision", "matplotlib", "tqdm"])


## 2) 导入库与训练配置
这段代码设置随机种子、设备（CPU/GPU）和基础超参数，保证实验可复现并可直接训练。


In [ ]:
if DEPENDENCIES_READY:
    import random
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    import matplotlib.pyplot as plt
    from tqdm.auto import tqdm

    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"????: {device}")

    batch_size = 128
    lr = 1e-3
    num_epochs = 5
    num_steps_sample = 100
else:
    print("???????????????????????")


## 3) 加载 MNIST 数据
这段代码下载并加载 MNIST 数据集，并把图像归一化到 `[-1, 1]`，便于与高斯噪声空间做插值训练。


In [ ]:
if DEPENDENCIES_READY:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_set = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)

    print(f"?????: {len(train_set)}")


## 4) 构建 Flow Matching 模型
这段代码定义一个带时间嵌入的 MLP 速度场网络 `v_theta(x,t)`，输入为当前状态和时间，输出与输入同维度的速度向量。


In [ ]:
if DEPENDENCIES_READY:
    class TimeEmbedding(nn.Module):
        def __init__(self, dim=64):
            super().__init__()
            self.dim = dim

        def forward(self, t):
            half = self.dim // 2
            freqs = torch.exp(
                torch.linspace(np.log(1.0), np.log(1000.0), half, device=t.device)
            )
            args = t[:, None] * freqs[None, :]
            return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


    class FlowMLP(nn.Module):
        def __init__(self, x_dim=28 * 28, t_dim=64, hidden=512):
            super().__init__()
            self.time_emb = TimeEmbedding(t_dim)
            self.net = nn.Sequential(
                nn.Linear(x_dim + t_dim, hidden),
                nn.SiLU(),
                nn.Linear(hidden, hidden),
                nn.SiLU(),
                nn.Linear(hidden, x_dim),
            )

        def forward(self, x, t):
            t_feat = self.time_emb(t)
            h = torch.cat([x, t_feat], dim=1)
            return self.net(h)


    model = FlowMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    print(f"?????: {sum(p.numel() for p in model.parameters()):,}")


## 5) 训练 Flow Matching
这段代码使用标准 Flow Matching 目标：采样噪声 `x0` 与真实图像 `x1`，构造 `xt=(1-t)x0+tx1`，并用 `x1-x0` 监督速度场。


In [ ]:
if DEPENDENCIES_READY:
    model.train()
    for epoch in range(num_epochs):
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        total_loss = 0.0
        for x, _ in pbar:
            x1 = x.to(device).view(x.size(0), -1)
            x0 = torch.randn_like(x1)
            t = torch.rand(x1.size(0), device=device)

            xt = (1.0 - t[:, None]) * x0 + t[:, None] * x1
            target_v = x1 - x0

            pred_v = model(xt, t)
            loss = F.mse_loss(pred_v, target_v)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        print(f"Epoch {epoch+1}: mean loss = {total_loss / len(train_loader):.4f}")


## 6) 采样生成手写数字
这段代码从高斯噪声出发，用欧拉法积分 `dx/dt=v_theta(x,t)`，把噪声逐步推到数据分布并可视化结果。


In [ ]:
if DEPENDENCIES_READY:
    @torch.no_grad()
    def sample_flow(model, n_samples=64, steps=100):
        model.eval()
        x = torch.randn(n_samples, 28 * 28, device=device)
        dt = 1.0 / steps
        for i in range(steps):
            t = torch.full((n_samples,), i / steps, device=device)
            v = model(x, t)
            x = x + dt * v
        x = x.view(-1, 1, 28, 28)
        x = x.clamp(-1, 1)
        x = (x + 1) / 2
        return x.cpu()


    samples = sample_flow(model, n_samples=64, steps=num_steps_sample)

    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i, 0], cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 7) 保存模型（可选）
这段代码将训练好的 Flow Matching 模型参数保存到本地，便于后续继续训练或单独采样。


In [ ]:
if DEPENDENCIES_READY:
    save_path = "flow_matching_mnist.pt"
    torch.save(model.state_dict(), save_path)
    print(f"??????: {save_path}")
